In [17]:
# wonderbk.com
# For books - https://wonderbk.com/
# For reviews - https://www.goodreads.com/
# How to get the description from the website
# https://www.wonderbk.com/shop/product/479822-jewish-holiday-style

In [1]:
import requests
from bs4 import BeautifulSoup
import json

# Scraping reivews

In [66]:
def review_scrap(url: str):
    """
        Parameter:
            url: A url for the book review are found
        
        Return:
            reviews: List of reviews 
            length: Legnth of the reviews list =
    """

    # This is for changing the response from 403(foribben) to 200 success
    headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.107 Safari/537.36"
    }
    
    if url:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            reviews = [i.text for i in soup.find_all('span', class_='Formatted')]
            
            return reviews, len(reviews)
    else:
        return "" ""

# Scraping Description

In [67]:
def scrap_details(url:str):

    """
        Parameter:
            url: A url for the details of the books such as book details (author, description, and more....)
        
        Return:
            Author: str
            Description of the book: str
            isbn number: number 
            review link: url link

    """
    
    response = requests.get(url)
    if response.status_code == 200:
        
        soup = BeautifulSoup(response.text, 'html.parser')

        author = soup.find('span', class_='author vcard')
    
        description = soup.find('div', class_='entry-description')

        isbn_details = soup.find("p", class_ = 'entry-details-isbn')

        reviews_link = soup.find('a', rel="nofollow")

        if author == None:
            author = None
        else:
            author = author.find('a').text 

        if description == None:
            description = None
        else:
            description = description.text.strip().replace("Read More", "").strip()

        if isbn_details == None:
            isbn_d = None
        else:
            isbn_d = "&_-_&".join(isbn_details.text.strip().replace(" ", "").split("\n"))
            # isbn_d = r.split("-")[0] + r.split("-")[1]
            # date = r.split("-")[2]

        src_link = reviews_link['href'] if reviews_link else None
            
        return author, description, isbn_d, src_link

In [68]:
def search_ss(url: str):
    # Send a GET request to fetch the content of the page
    response = requests.get(url)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')

        books = soup.find_all('h3', class_='entry-title')

        # Find the <a> tag with class "entry-link"
        a_tag = soup.find_all('a', class_='entry-link')

        books_data = []
        
        for ix, vl in enumerate(a_tag):
            # books_data.append(json.loads(a_tag[ix].get('data-click-impression')))

            book_title = json.loads(a_tag[ix].get('data-click-impression'))["ecommerce"]["items"][0]["item_name"]
            book_price = json.loads(a_tag[ix].get('data-click-impression'))["ecommerce"]["items"][0]["price"]
            book_genre = url.split("/")[5]
            book_sub_genre = url.split("/")[6]
            book_details_link = a_tag[ix].get('href')

            author, description, isbn_d, src_link = scrap_details(book_details_link)
            
            reviews, num_reviews = review_scrap(src_link)

            book_details = {
                        "book_title": book_title,
                        "author": author,
                        # "date": date,
                        "description": description,
                        "isbn_details": isbn_d,
                        "book_price": book_price,
                        "book_genre": book_genre,
                        "book_sub_genre": book_sub_genre,
                        "book_details_link": book_details_link,
                        "reviews": reviews,
                        "num_reviews": num_reviews

                    }
        
            books_data.append(book_details)

        return books_data
    

# Genre

In [69]:
list_genres = [
    {
       "art": ["african", "annuals", "art-and-politics", "australian-and-oceanian", "canadian", "ceramics", "color-theory", "body-art-and-tattoing", "business-ascpects", "canadian", "caribean-and-latin-america"] 
    },
    {
     "drama":["african", "canadian", "ancient-and-classical", "ancient--classical-and-medieval", "anthologies"]   
    },
    {
    "education":["adult-and-continuing-education", "aims-and-objectives"]
    }
]

# Scraping the books

In [71]:
all_books = []
for i in list_genres:
    for ix, vl in i.items():
        for j in vl:
            # https://www.wonderbk.com/shop/books/education/elementary
            rs = search_ss(f"https://www.wonderbk.com/shop/books/{ix}/{j}")
            if rs is not None:
                for z in rs:
                    all_books.append(z)

In [72]:
print(f"The number of books are {len(all_books)}")

The number of books are 226


In [73]:
all_books

[{'book_title': 'Quill and Beadwork of the Western Sioux',
  'author': 'Lyford, Carrie A.',
  'description': None,
  'isbn_details': 'ISBN:0933472005/&_-_&Publisher:JohnsonPub.Co,&_-_&January1979',
  'book_price': '5.69',
  'book_genre': 'art',
  'book_sub_genre': 'african',
  'book_details_link': 'https://www.wonderbk.com/shop/product/932883-quill-and-beadwork-of-the-western-sioux',
  'reviews': ['The Sioux believed that the art of quilling was brought to a woman in a dream. She in turn taught others how to work with the quills. Each woman used personal and original designs which she received through her own dreams. When European beads became available the traditional designs were carried over into beadwork. Quill and Beadwork of the Wesern Sioux is the classic work on these important Plains Indian arts, applying to the Cheyennes and Arapahoes as well as to the Sioux. The book is at once a history and a practical, easy to follow manual for people wishing to recreate for themselves obj

In [ ]:
# Convert the list into a csv file
import csv
import os
os.makedirs("./dataset")
csv_file = "./dataset/books_dataset.csv"

# Writing to CSV file
with open(csv_file, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=all_books[0].keys())
    
    # Write the header
    writer.writeheader()
    
    # Write the data
    writer.writerows(all_books)

print(f"CSV file '{csv_file}' created successfully!")

CSV file './dataset/books_dataset.csv' created successfully!


# Scrap the descriptions

In [142]:
url_dt = "https://www.wonderbk.com/shop/product/1564647-the-hare-with-amber-eyes-a-hidden-inheritance"

In [143]:

response = requests.get(url_dt)
if response.status_code == 200:
    # Parse the page content using BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')
    # Find all book containers (assumed that they are in an article tag with class 'product_pod')
    author = soup.find('span', class_='author vcard')
    # Find the <a> tag with class "entry-link"
    description = soup.find('div', class_='entry-summary')

    isbn_details = soup.find("p", class_ = 'entry-details-isbn')

#    author[0].find('a').text, description[0].text.strip().replace("Read More", "").strip()

# Viewing the book dataset csv

In [1]:
import pandas as pd
book_ds = pd.read_csv("./dataset/books_dataset.csv")
book_ds.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226 entries, 0 to 225
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   book_title         226 non-null    object 
 1   author             226 non-null    object 
 2   description        176 non-null    object 
 3   isbn_details       226 non-null    object 
 4   book_price         226 non-null    float64
 5   book_genre         226 non-null    object 
 6   book_sub_genre     226 non-null    object 
 7   book_details_link  226 non-null    object 
 8   reviews            226 non-null    object 
 9   num_reviews        226 non-null    int64  
dtypes: float64(1), int64(1), object(8)
memory usage: 17.8+ KB


In [2]:
book_ds

,book_title,author,description,isbn_details,book_price,book_genre,book_sub_genre,book_details_link,reviews,num_reviews
0,Quill and Beadwork of the Western Sioux,"Lyford, Carrie A.",NaN,"ISBN:0933472005/&_-_&Publisher:JohnsonPub.Co,&...",5.69,art,african,https://www.wonderbk.com/shop/product/932883-q...,['The Sioux believed that the art of quilling ...,3
1,PARALELL: HANNA COLLINS,"Shulman, Ken",NaN,"ISBN:8496954382/&_-_&Publisher:ACTAR,LaboralCi...",37.47,art,african,https://www.wonderbk.com/shop/product/1663275-...,['Originating as an installation of three simu...,3
2,Royal Benin Art in the Collection of the Natio...,National Museum of African Art (U. S.) (COR),"Shows pendants, spoons, figurines, and plaques...","ISBN:0874744458/&_-_&Publisher:Smithsonian,&_-...",20.77,art,african,https://www.wonderbk.com/shop/product/1239577-...,"['Shows pendants, spoons, figurines, and plaqu...",4
3,"Visual Arts of Africa: Gender, Power, and Life...","Perani, Judith","Presented by geographic region, this book prov...","ISBN:0134423283/&_-_&Publisher:Pearson,&_-_&Ju...",19.05,art,african,https://www.wonderbk.com/shop/product/3722516-...,"['Presented by geographic region, this book pr...",3
4,Desert Jewels: North African Jewelry and Photo...,"Loughran, Kristyne",Desert Jewels presents a superb collection of ...,ISBN:0945802528/&_-_&Publisher:MuseumforAfrica...,121.82,art,african,https://www.wonderbk.com/shop/product/3806737-...,['Desert Jewels presents a superb collection o...,4
...,...,...,...,...,...,...,...,...,...,...
221,How Can I Fix It?: Finding Solutions and Manag...,"Cuban, Larry",NaN,ISBN:0807740497/&_-_&Publisher:TeachersCollege...,12.99,education,aims-and-objectives,https://www.wonderbk.com/shop/product/3414997-...,['With this highly accessible and unique littl...,4
222,The Bridge to Brilliance: How One Principal in...,"Paley, Rebecca (CON)",Be inspired by the magnetic young principal wh...,"ISBN:1101980257/&_-_&Publisher:Viking,&_-_&Aug...",6.03,education,aims-and-objectives,https://www.wonderbk.com/shop/product/3574586-...,['Be inspired by the magnetic young principal ...,33
223,Dream Differently: Candid Advice for America's...,"Bertram, Vince M., Dr.","To get the most out of your college education,...","ISBN:1621576787/&_-_&Publisher:Regnery,&_-_&Au...",5.69,education,aims-and-objectives,https://www.wonderbk.com/shop/product/3628320-...,"[""To get the most out of your college educatio...",3
224,Most Likely to Succeed: Preparing Our Kids for...,"Wagner, Tony",From two leading experts in education and entr...,"ISBN:1501104314/&_-_&Publisher:Scribner,&_-_&A...",5.69,education,aims-and-objectives,https://www.wonderbk.com/shop/product/4076792-...,['From two leading experts in education and en...,33
